In [ ]:
"""
Custom CNN - Binary Image Classification (Cats vs Dogs)
Beginner-friendly, well-commented training script.
"""

import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

# ------------------------------
# STEP 0: SETTINGS
# ------------------------------
NUM_CLASSES = 2          # binary: cats vs dogs
IMG_SIZE = (128, 128)
BATCH_SIZE = 16
EPOCHS = 20

TRAIN_DIR = "/content/dataset/train"   # contains cats/ and dogs/
TEST_DIR  = "/content/dataset/test"    # contains cats/ and dogs/

# ------------------------------
# STEP 1: LOAD DATA
# ------------------------------
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

class_names = train_ds.class_names
print("Classes found:", class_names)   # ['cats', 'dogs']

# ------------------------------
# STEP 2: PREPROCESSING
# ------------------------------
normalization_layer = layers.Rescaling(1./255)

data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds  = test_ds.map(lambda x, y: (normalization_layer(x), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds  = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

# ------------------------------
# STEP 3: BUILD THE CNN
# ------------------------------
model = models.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),

    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(2, 2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(1, activation='sigmoid')   # binary output
])

# ------------------------------
# STEP 4: COMPILE
# ------------------------------
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ------------------------------
# STEP 5: TRAIN
# ------------------------------
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS
)

# ------------------------------
# STEP 6: EVALUATE ON TEST SET
# ------------------------------
test_loss, test_acc = model.evaluate(test_ds)
print(f"\nTest Accuracy: {test_acc*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

# ------------------------------
# STEP 7: PLOT ACCURACY & LOSS
# ------------------------------
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss')

plt.tight_layout()
plt.savefig("training_curves.png", dpi=120)
print("Saved training_curves.png")

# ------------------------------
# STEP 8: SAVE MODEL
# ------------------------------
model.save("custom_cnn_catsdogs.keras")
print("Model saved as custom_cnn_catsdogs.keras")

Found 557 files belonging to 2 classes.
Found 140 files belonging to 2 classes.
Classes found: ['cats', 'dogs']


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_15 (Conv2D)              │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_15 (MaxPooling2D) │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_16 (MaxPooling2D) │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_17 (MaxPooling2D) │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │     4,194,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,287,809 (16.36 MB)

 Trainable params: 4,287,809 (16.36 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 36s 973ms/step - accuracy: 0.5171 - loss: 0.7101 - val_accuracy: 0.4500 - val_loss: 0.6928
Epoch 2/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 24s 672ms/step - accuracy: 0.5153 - loss: 0.6927 - val_accuracy: 0.4786 - val_loss: 0.6929
Epoch 3/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 23s 670ms/step - accuracy: 0.5422 - loss: 0.6922 - val_accuracy: 0.5857 - val_loss: 0.6861
Epoch 4/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 49s 922ms/step - accuracy: 0.5889 - loss: 0.6719 - val_accuracy: 0.5500 - val_loss: 0.6678
Epoch 5/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 23s 657ms/step - accuracy: 0.6104 - loss: 0.6611 - val_accuracy: 0.5643 - val_loss: 0.6610
Epoch 6/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 25s 720ms/step - accuracy: 0.6373 - loss: 0.6312 - val_accuracy: 0.5786 - val_loss: 0.6690
Epoch 7/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 22s 641ms/step - accuracy: 0.6732 - loss: 0.6000 - val_accuracy: 0.6500 - val_loss: 0.6851
Epoch 8/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 24s 696ms/step - accuracy: 0.7235 - loss: 0.5436 - val_accu

In [ ]:
"""
Custom CNN - Binary Image Classification (Cats vs Dogs)
Beginner-friendly, well-commented training script.
"""

import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

# ------------------------------
# STEP 0: SETTINGS
# ------------------------------
NUM_CLASSES = 2          # binary: cats vs dogs
IMG_SIZE = (128, 128)
BATCH_SIZE = 16
EPOCHS = 100

TRAIN_DIR = "/content/dataset/train"   # contains cats/ and dogs/
TEST_DIR  = "/content/dataset/test"    # contains cats/ and dogs/

# ------------------------------
# STEP 1: LOAD DATA
# ------------------------------
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

class_names = train_ds.class_names
print("Classes found:", class_names)   # ['cats', 'dogs']

# ------------------------------
# STEP 2: PREPROCESSING
# ------------------------------
normalization_layer = layers.Rescaling(1./255)

data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds  = test_ds.map(lambda x, y: (normalization_layer(x), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds  = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

# ------------------------------
# STEP 3: BUILD THE CNN
# ------------------------------
model = models.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),

    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(2, 2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(1, activation='sigmoid')   # binary output
])

# ------------------------------
# STEP 4: COMPILE
# ------------------------------
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ------------------------------
# STEP 5: TRAIN
# ------------------------------
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS
)

# ------------------------------
# STEP 6: EVALUATE ON TEST SET
# ------------------------------
test_loss, test_acc = model.evaluate(test_ds)
print(f"\nTest Accuracy: {test_acc*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

# ------------------------------
# STEP 7: PLOT ACCURACY & LOSS
# ------------------------------
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss')

plt.tight_layout()
plt.savefig("training_curves.png", dpi=120)
print("Saved training_curves.png")

# ------------------------------
# STEP 8: SAVE MODEL
# ------------------------------
model.save("custom_cnn_catsdogs.keras")
print("Model saved as custom_cnn_catsdogs.keras")